# Forest Disturbance Monitoring with Geospatial Foundation Models

Hands-on notebook for the [**2nd ESA-NASA Workshop on AI Foundation Models for
Earth Observation**](https://nikal.eventsair.com/2nd-esa-nasa-workshop-on-ai-foundation-model-for-earth-observation-eo).

> **⚠️ Disclaimer.** This notebook and dataset come from an ongoing research project and
> **should not be redistributed**.
> A larger, non-anonymised dataset and benchmark will be released later this year.
> Reach out to **damien.robert@uzh.ch** if you would like to know more.

#### **🌲 What's our goal?**

Our focus is on **near real-time forest disturbance monitoring**. That is, given series of
Sentinel-2 satellite observations, we want to detect anomalies happening in the forests
and identify their **_agent_** (i.e. what caused them), as early as possible. 

We will work with a dataset of 200 cloud-free Sentinel-2 time-series patches. 
For each of these, we have access to human annotations indicating whether a disturbance 
event happened and when. Importantly, the annotations **_only cover the center pixel_**, the 
state of the rest of the pixels is **_unknown_**.

We will build simple models to explore two tasks of interest for forestry applications:

1. **Detection** — _"given current and past observations, is the forest currently disturbed?"_
2. **Attribution** — _"given observations around a known disturbance event, which agent caused it?"_

These models will necessarily need input **_features_** coming from our image time series.
In this notebook, we will be exploring how TerraTorch can help us extract useful expressive 
representations from the images with minimal effort, and comparing how these so-called **_embedddings_**
fare against more traditional handcrafted descriptors.

> The notebook is one continuous narrative. Cells labelled **👩‍💻 Exercise** are
> *non-blocking* — they invite you to swap a knob and re-run, but everything
> runs as-is, so the notebook never gets stuck on an unfilled exercise.

#### **🤖 Why foundation models for forest disturbance monitoring?**

Earth Observation (EO) data from Sentinel-2 allow to monitor
forests at high spatio-temporal resolution. In practice, however,
exploiting that data still requires deep domain expertise, large data
transfers, access to HPC, and ML skills. These barriers keep
EO-based analytics out of reach for many downstream applications.

**Geospatial Foundation Models** (GFMs) offer a way around this: pre-trained
on huge volumes of unlabeled EO data, GFMs learn general-purpose
representations that decouple representation-learning from downstream-task
modelling. Users can plug into expressive features from modern deep nets
with limited EO or ML expertise and modest compute. This notebook
demonstrates an **embedding-as-a-service** workflow: embeddings are
pre-computed once server-side (here with frozen TerraMind-small), shipped
as lightweight tensors, and consumed by simple downstream models — linear
probes, small MLPs — that you can train on your laptop.

We target **forest disturbance monitoring**, where *timely* inference and
regional customisation often hinder adoption of involved deep learning-based approaches.
Disturbances such as logging, windthrow, fire, pests and diseases can occur
abruptly and require rapid detection to support conservation, policy, and
risk management. Existing operational tools (e.g. those aggregated by 
[Global Forest Watch](https://www.globalforestwatch.org/map/) typically rely on region- and sensor-specific models with
limited feature expressivity, and so may benefit from the richer multi-modal
and spatio-temporal representations captured by GFMs — provided those
embeddings can be reached through a practical deployment pipeline. This
notebook explores the construction of a concrete instance of such pipeline.

#### **🖥️ How do embeddings come into play in this notebook?**

The motivation behind this notebook, is to illustrate how an infrastructure distributing on-demand, 
precomputed embeddings to the user could be used for near real-time forest monitoring. 
Creating the building blocks of such an infrastucture is the main focus of the 
[Embed2Scale](https://embed2scale.eu/) project.

For simplicity here, we will assume already-computed embeddings, which are provided for this workshop.

```text
embedding-as-a-service workflow
└── server side:  EO data → frozen FM → embeddings (zarr)
└── client side:  embeddings → lightweight feature construction → simple linear/MLP head
```

## 0. Setup

Standard imports, data paths, device detection.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import warnings
from pathlib import Path

import datetime as dt
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import torch
import zarr

# Silence the "CUDA driver too old" UserWarning some workshop images
# emit when torch is built against CUDA 13 but the host driver is
# older. s2tutorial.DEVICE probes CUDA defensively and falls back to
# CPU automatically when this happens.
warnings.filterwarnings(
    "ignore",
    message=r".*CUDA initialization.*",
    category=UserWarning,
)

import s2tutorial as s2t
from s2tutorial import TorchClassifier

# ── Workshop data location ─────────────────────────────────────────
# The workshop dataset (~15 GB: patches + embeddings + audit) ships as
# a separate bundle, NOT inside this repo. Resolution order:
#   1. $WORKSHOP_DATA_ROOT             (env var — wins if set)
#   2. SageMaker default unzip path    (auto-detected on the workshop VM)
#   3. ../data                         (local sibling for laptop dev)
# Both local filesystem paths and `s3://bucket/...` URIs are accepted.
WORKSHOP_DATA_ROOT_SAGEMAKER = "/home/sagemaker-user/workshop_data/workshop_bundle/data"
WORKSHOP_DATA_ROOT_LOCAL     = "../data"

_env_root = os.environ.get("WORKSHOP_DATA_ROOT")
if _env_root:
    _root_str = _env_root
elif Path(WORKSHOP_DATA_ROOT_SAGEMAKER).exists():
    _root_str = WORKSHOP_DATA_ROOT_SAGEMAKER
else:
    _root_str = WORKSHOP_DATA_ROOT_LOCAL

ROOT = _root_str if "://" in _root_str else Path(_root_str)
# audit/ sits as a sibling of data/ inside the unzipped bundle. Must be
# a local path (numpy.load can't seek over the network); override via
# WORKSHOP_AUDIT_DIR if you keep the audit files elsewhere.
AUDIT_DIR = Path(
    os.environ.get(
        "WORKSHOP_AUDIT_DIR",
        "../audit" if "://" in _root_str else str(Path(_root_str).parent / "audit"),
    )
)

# Dataset-wide PCA + RGB stretch are precomputed once and shipped in
# audit/. PCA_STATE is referenced again in Section 1 with an in-context
# explainer; load here so it is available everywhere.
PCA_STATE = np.load(AUDIT_DIR / "token_pca.npz")

# Default to CPU for portability. Set DEVICE = s2t.DEVICE (or
# torch.device("cuda")) if your machine has a CUDA-13-compatible
# driver and you want GPU acceleration on dense-prediction cells.
DEVICE = torch.device("cpu")
print(f"Torch device:   {DEVICE}  (s2t.DEVICE auto-detected: {s2t.DEVICE})")
meta = s2t.load_metadata(ROOT)
print(
    f"Loaded {len(meta['samples'])} samples, "
    f"{len(meta['frames'])} S2 frames, "
    f"{len(meta['labels'])} label rows from {ROOT}"
)

# Four example samples — one per disturbance class. Each figure section
# below picks one as its active example; swap by re-assigning
# EXAMPLE_SID at the top of that section.
EXAMPLE_CLEAR_SID = 29   # 211 — Clear-Cut
EXAMPLE_THIN_SID  = 31   # 212 — Thinning
EXAMPLE_FIRE_SID  = 44   # 242 — Wildfire
EXAMPLE_WIND_SID  = 23    # 243 — Windthrow

## 1. Data exploration

<p align="center">
  <img src="assets/disturbances.png" alt="Forest disturbance examples" width="800">
</p>

Our dataset targets four forest disturbance agents: 
- **Clear-cut** (complete removal of canopy - label 211)
- **Thinning** (partial removal of canopy - label 212)
- **Wildfire** (burn scars - label 242)
- **Windthrow** (storms tearing canopy - label 243)

The dataset is made of **200 time series** of Sentinel-2 observations of 252x252 pixels.
For each time series, we only have annotations concerning the **center pixel** at 
`(126, 126)`, the other pixels are considered non-annotated.
Each time series contains only a **single disturbance event**.
The annotations indicate **when** this disturbance happened and **which agent** caused it.

### Dataset stats

Let's load general information about the dataset.

In [ ]:
samples = meta["samples"]
labels = meta["labels"]
frames = meta["frames"]

# Temporal span: span of clear S2 acquisition dates across all 200 samples.
dates_year = frames["date"].cast(pl.Date).dt.year()
year_min, year_max = int(dates_year.min()), int(dates_year.max())

# S2 patch geometry — read one zarr to confirm shape + resolution.
g0 = zarr.open_group(str(ROOT / "patches" / f"{int(samples['sample_id'][0])}.zarr"),
                     mode="r")
T0, _, H, W = g0["s2_10m"].shape   # (T, 4 bands, H, W)

print(f"Samples (time-series):    {len(samples)}")
print(f"S2 cloud-free frames:     {len(frames)} "
      f"(median per sample = {frames.group_by('sample_id').len()['len'].median():.0f})")
print(f"Temporal span:            {year_min} – {year_max}  "
      f"({samples['window_start'].min()} → {samples['window_end'].max()})")
print(f"S2 patch size (10 m):     {H} × {W} px  ({H * 10} m × {W * 10} m on ground)")
print(f"S2 native band groups:    10 m (B02, B03, B04, B08), 20 m, 60 m, SCL")
print(f"Disturbance events:       {len(labels.filter(pl.col('label').is_in(list(s2t.DISTURBANCE_CODES))))}")
print()

fig = s2t.event_class_breakdown(meta)
plt.show()

### One example time-series

Now, let's see what a sample time series looks like. Feel free to play with 
`EXAMPLE_SID` and `n_patches` in the figure below.

In [ ]:
# Switch the example by editing this single line.
EXAMPLE_SID = EXAMPLE_FIRE_SID   # try EXAMPLE_WIND_SID / EXAMPLE_THIN_SID / EXAMPLE_CLEAR_SID / or anything in [0, 199]

ts_example = s2t.get_sample(ROOT, EXAMPLE_SID, metadata=meta)
ev = ts_example.event_period()

fig = s2t.sample_timeline(
    ts_example,
    n_patches=8,         # -1 = all frames, 0 = swim-lane only
    crop_size=None,      # adjust crop_size and crop_offset to zoom in 
    crop_offset=(0, 0),  # adjust crop_size and crop_offset to zoom in
    mode="rgb",          # 'rgb' / 'scl' / 'both'
    fig_width=14.0,      # adjust to taste
)
plt.show()

### 👩‍💻 Exercise — non-blocking

Visualize the entire time series of observations at once with `n_patches=-1`. 
This illustrates how unevenly the satellite revisits a single location 
(gaps in winter, dense summer coverage, occasional missing months).

Play with `crop_size` and `crop_offset` to zoom in on the images.

In [ ]:
fig = s2t.sample_timeline(
    ts_example,
    n_patches=-1,       # all frames
    crop_size=64,      # adjust crop_size and crop_offset to zoom in 
    crop_offset=(0, 0),  # adjust crop_size and crop_offset to zoom in
    mode="rgb",
    fig_width=14.0,
)
plt.show()

## 2. Extracting image features

Now we have had a look at our images, we need to define ways of extracting 
information from those for downstream analysis. In this notebook, we want to 
compare how a "traditional" **🛠️ handcrafted** (baseline) features perform against 
**🤖 foundation model embeddings** (FM). 
Both feature paths emit **one vector per frame** at the **labelled centre of the patch**.

> To save time, we have precomputed all necessary per-frame features for you. These are
> distributed in the workshop's data bundle.

In this section, we will detail how each feature set is obtained.

In [ ]:
_pix_pack = np.load(AUDIT_DIR / "center_pixel_features.npz", allow_pickle=True)
_pix_sids = _pix_pack["sids"].tolist()
_pix_offsets = _pix_pack["offsets"]
_pix_feats = _pix_pack["feats"]
_pix_index = {int(s): k for k, s in enumerate(_pix_sids)}
print(f"Baseline pre-computed: {_pix_feats.shape[0]} frames × "
      f"{_pix_feats.shape[1]} features  ({_pix_feats.nbytes / 1e6:.1f} MB)")


def emb_centre_for(sid: int) -> np.ndarray:
    g = zarr.open_group(
        str(s2t.default_emb_center_root(ROOT) / f"{sid}.zarr"), mode="r",
    )
    return np.asarray(g["embedding"][:]).astype(np.float32)        # (T, 384)


def pix_centre_for(sid: int) -> np.ndarray:
    k = _pix_index[int(sid)]
    return _pix_feats[_pix_offsets[k]:_pix_offsets[k + 1]]         # (T, 15)

### 🛠️ Handcrafted features - _Baseline_

At the single annotated centre pixel of the 252 × 252
footprint (pixel `(126, 126)`), every frame produces a **15-d vector**:
the 12 stored Sentinel-2 spectral bands (B01–B12, the cirrus band B10 is dropped
by ESA at L2A) plus three normalised vegetation indices:

  $$\text{NDVI} = \frac{B_{08}-B_{04}}{B_{08}+B_{04}} \qquad
    \text{NDMI} = \frac{B_{08}-B_{11}}{B_{08}+B_{11}} \qquad
    \text{NDWI} = \frac{B_{03}-B_{08}}{B_{03}+B_{08}}$$

The 12 raw Sentinel-2 bands are spectral reflectances — not interpretable on their
own. The three normalised indices are the part of the baseline vector that
carries explicit domain knowledge:

- **NDVI** — *vegetation greenness*. High over dense canopies; drops
  sharply when leaves are lost (logging, fire, drought, defoliation).
- **NDMI** — *canopy moisture*. High over well-watered forest; drops
  in dry / burnt scars and on bare soil.
- **NDWI** — *open-water presence*. Strongly negative over vegetation,
  positive over standing water and saturated ground.

Below we render these three indices at every pixel for **one example
frame** of our example sample. 

In [ ]:
# Switch the example by editing this single line.
EXAMPLE_SID = EXAMPLE_FIRE_SID   # try EXAMPLE_WIND_SID / EXAMPLE_THIN_SID / EXAMPLE_CLEAR_SID
ts_example = s2t.get_sample(ROOT, EXAMPLE_SID, metadata=meta)

# Pick the first post-event frame so the wildfire / windthrow / clear-cut
# signal is visible in the index maps. Re-assign EXAMPLE_FRAME_IDX to any
# other frame index in [0, len(ts_example)) to explore.
ev = ts_example.event_period()
EXAMPLE_FRAME_IDX = next(
    (i for i, d in enumerate(ts_example.dates)
     if dt.date.fromisoformat(d) > ev["start"]),
    len(ts_example) - 1,
)

fig = s2t.baseline_index_explainer(
    ts_example,
    EXAMPLE_FRAME_IDX,
    title=f"sid={EXAMPLE_SID} — frame {EXAMPLE_FRAME_IDX} "
          f"({ts_example.dates[EXAMPLE_FRAME_IDX]})",
)
plt.show()

Let's look at a time series of those handcrafted indices. For RGB visualization we use: NDVI → R, NDMI → G, NDWI → B.

In [ ]:
# Multi-frame composite — twin of `token_pca_grid` for the baseline path.
# Each panel on the lower row packs NDVI/NDMI/NDWI into one false-colour
# image (NDVI→R, NDMI→G, NDWI→B) with a fixed dataset-wide stretch, so
# colours can be compared across frames AND across samples.
fig = s2t.baseline_index_grid(
    ts_example,
    n_patches=8,        # -1 = all frames, 0 = swim-lane only
    fig_width=14.0,     # adjust to taste
    title=f"sid={EXAMPLE_SID} — handcrafted indices across the time series",
)
plt.show()

⚠️ Important: for the rest of this notebook, the models using these features will 
only ever consumes the values at the **centre pixel** (white circle); the dense maps are
purely there to build intuition about what each index measures and where
the disturbance signal will sit in the 15-d baseline vector.

### 🤖 Foundation Model features - _FM_

For this notebook, we pre-computed embeddings of our Sentinel-2 images using TerraTorch.
More specifically, we used the **TerraMind Small model**. 

**Token ↔ pixel alignment.**
As a reminder, the images in our dataset are 252×252, with the annotation at
pixel `(126, 126)`.
But TerraMind was trained to consume `(224, 224)` images and to output a
`(14, 14)` token grid of 384-d embeddings.
For this reason, we do not feed the entire 252x252 images to TerraMind.
Instead, we extract crops `[6:230, 6:230]`
(224×224). These are then tokenized by TerraMind using 16×16 ViT patches, 
yielding a 14×14 output token grid. The centre token `(7, 7)` covers
original-patch pixels `[118:134, 118:134]` and is well-centered over the
original annotated pixel `(126, 126)`.
Said otherwise, the `(7, 7)` token is the one carrying our disturbance 
annotations.

<p align="center">
  <img src="assets/image_crop_and_tokenize.png" alt="Patch / crop / token grid hierarchy" width="640">
</p>

Whenever we want to visualize embeddings, because we cannot visualize 384-d easily,
we project these tokens to their first 3 PCA components and view them as a
false-colour image. **Crucial detail:** the PCA fit is **dataset-wise**, not
per-frame. This means that the same colour means same direction in token
space — so colours can be compared *across* frames and *across* samples.

The PCA basis lives in `audit/token_pca.npz` and was loaded in Section 0.
We use it everywhere a "shared embedding direction" is meaningful — across
frames of the same sample, and across different samples.

Let's visualize a time series of token embeddings now!

In [ ]:
# The state is already loaded at the top of the notebook. Re-print here
# so the variance-explained ratio is visible in context.
print(f"components shape: {PCA_STATE['components'].shape}")
print(f"mean shape:       {PCA_STATE['mean'].shape}")
print(f"variance explained per component: {', '.join(f'{v:.2f}' for v in PCA_STATE['explained_variance_ratio'])}")

In [ ]:
# 6 evenly-spaced frames spanning before / event / after.
event_date = ts_example.event_period()["start"]
dates_dt = [dt.date.fromisoformat(d) for d in ts_example.dates]
pre_idx = [i for i, d in enumerate(dates_dt) if d < event_date]
post_idx = [i for i, d in enumerate(dates_dt) if d >= event_date]
def _evenly(seq, n):
    if len(seq) <= n: return list(seq)
    step = (len(seq) - 1) / (n - 1)
    return [seq[round(i * step)] for i in range(n)]
showcase_idx = _evenly(pre_idx, 3) + _evenly(post_idx, 3)

fig = s2t.token_pca_grid(
    ts_example,
    pca_state=PCA_STATE,
    frame_indices=showcase_idx,
    title=f"sid={EXAMPLE_SID} — top: RGB (224 px crop), bottom: dataset-PCA tokens",
)
plt.show()

Now we have covered the image feature extraction, let's see how to detect forest disturbances with these!

## 3. Disturbance monitoring intuition

Forest reflectance and structure follow strong **annual cycles**: leaf-out,
senescence, snow cover, sun-angle drift. A query observation compared to its
immediately preceding one may simply differ because of weather and
phenological variation between the two acquisition dates.

Instead, we compare each query observation to the **same calendar window in
past years** at the same place — same time-of-year, same sun angle, similar
phenology. The difference between query and references is then much more
specifically about *what changed in the meantime*: if there has been a
structural disturbance, the query sits off-manifold; if not, query and
references are on the same seasonal loop.

To get a (rough) sense of how the precomputed embeddings behave around a disturbance, 
we are goind to visualize the evolution of our `(7, 7)` token 
of interest, along its first 2 principal components.

Our plot has two side-by-side views of the same trajectory:
- **Left**: markers are coloured by **month of year**,
  illustrating seasonal patterns.
- **Right**: markers are coloured by **period**
  (before / after the disturbance event), illustrating jumps in the signal
  caused by disturbances.

Use the **slider** below the figure to control the passing of time.

In [ ]:
# Switch the example by editing this single line.
EXAMPLE_SID = EXAMPLE_FIRE_SID   # try EXAMPLE_WIND_SID / EXAMPLE_THIN_SID / EXAMPLE_CLEAR_SID
ts_example = s2t.get_sample(ROOT, EXAMPLE_SID, metadata=meta)

s2t.embedding_trajectory(
    ts_example,
    PCA_STATE,
    coord=(7, 7),  # token coordinate
    figsize=(13, 5.3),     # adjust to taste
    title=f"sid={EXAMPLE_SID} — embedding-space trajectory (dataset-PCA basis)",
)

Hopefully, you should now get a sense that these **embeddings are carrying useful information for forest disturbance monitoring**.
Now we need to build models to make practical use of these embeddings.

## 4. Task A — NRT Detection

<p align="center">
  <img src="assets/nrt_detection.png" alt="NRT detection recipe" width="80%">
</p>

#### 🎯 Detection task
For every observation, decide _"is the forest
disturbed right now?"_ by comparing the query to its same-calendar-period
references in the past. 

Concretely, for each frame, we define a **📆 Reference set R**: 
past frames of the same sample within
`±REF_WINDOW_DAYS` of the query's calendar (month, day), over the past
`N_REF_YEARS` years (excluding the current one). Require at least
`MIN_REF_FRAMES` references — drop queries that don't qualify.

The model will have to classify the current observation either or two classes:
- **🔴 Disturbed** = **Positive label** = query date falls within
  `[event_start, event_start + ALERT_WINDOW_MONTHS]` (12 months by
  default — long enough to also cover the noisy revegetation phase).
- **🟢 Undisturbed** = **Negative label** = query date is anywhere else in the time series.

**📏 Metric**: this is a binary classification task with class imbalance. We will measure 
performance by macro-averaged **F1 score**.

To do this task, our models will leverage our **per-frame features** (🛠️ handcrafted or 🤖 embeddings).
In order to combine the features from the query and the reference set, we will
try various **_feature recipes_**. We propose several approaches in 
`s2t.RECIPE_REGISTRY` that you can try out.

### Temporal features recipes

The detection head sees a fixed-length **feature vector** built from two
ingredients: the query observation $q \in \mathbb{R}^D$ and the reference
set $R \in \mathbb{R}^{N \times D}$ (rows = past frames at the same
calendar period). We notate:

- $\bar{R} \;=\; \tfrac{1}{N}\sum_{r \in R} r$ — mean reference vector
- $\sigma_{R} \;=\; \mathrm{std}(R)$ along the row axis — per-feature spread
- $\mathrm{CD}(a, b) \;=\; 1 - \tfrac{a \cdot b}{\lVert a \rVert\, \lVert b \rVert}$ — cosine distance (0 = same direction, 1 = orthogonal, 2 = opposite)
- $\overline{\mathrm{CD}}_{R} \;=\; \tfrac{1}{N}\sum_{r \in R} \mathrm{CD}(r, \bar{R})$ — average cosine spread inside the reference set ("pseudo-std" in cosine space)

`s2t.RECIPE_REGISTRY` exposes six recipes; the active one is set by the
`RECIPE` knob below.

| name | formula | dim |
|---|---|---|
| `diff` | $q - \bar{R}$ | $D$ |
| `cd_scalar` | $\big[\mathrm{CD}(q, \bar{R})\big]$ | $1$ |
| `cd_pseudo_std` | $\big[\mathrm{CD}(q, \bar{R}),\ \overline{\mathrm{CD}}_{R}\big]$ | $2$ |
| `query_cd` | $\big[q,\ \mathrm{CD}(q, \bar{R}),\ \overline{\mathrm{CD}}_{R}\big]$ | $D + 2$ |
| `query_mean_cd` | $\big[q,\ \bar{R},\ \mathrm{CD}(q, \bar{R}),\ \overline{\mathrm{CD}}_{R}\big]$ | $2D + 2$ |
| `full` | $\big[q,\ \bar{R},\ q - \bar{R},\ \sigma_{R}\big]$ | $4D$ |

With $D = 384$ for FM (centre token) and $D = 15$ for the baseline
(centre pixel: 12 S2 bands + NDVI / NDWI / NDMI).

**How to read these.** `diff` and `cd_scalar` are the two extremes:
`diff` keeps every dimension of the change (large, sparse, needs more
training data), while `cd_scalar` distils the comparison to one scalar.
The `cd_*` recipes are cheap "pseudo-stat" features — useful as sanity
baselines. `query_cd` and `query_mean_cd` give the classifier the raw
query (and optionally the reference mean) on top of the change signal,
so it can also learn class-conditional absolutes. `full` concatenates
everything — broadest, highest-dim.

`full` is the default below; try changing `RECIPE` to one of the smaller
recipes to see how the train / val gap closes.

In [ ]:
# Tunable knobs for the detection task — edit and re-run the cells
# below to see the effect.
RECIPE          = "full"   # one of: "diff", "cd_scalar", "cd_pseudo_std",  "query_cd", "query_mean_cd", "full"
N_REF_YEARS     = 2        # number of past years contributing to R
REF_WINDOW_DAYS = 45       # +/- calendar window around the query date
ALERT_MONTHS    = s2t.ALERT_WINDOW_MONTHS   # length of the positive-label window

print(f"recipe = {RECIPE!r}")
print(f"features per query for FM      : {s2t.recipe_feature_dim(RECIPE, 384)}")
print(f"features per query for baseline: {s2t.recipe_feature_dim(RECIPE, 15)}")

### Illustration with $q - \bar{R}$

To illustrate the above ideas, let's have a look at the signal carried 
by $q - \bar{R}$ (ie the `diff` recipe). To this end, we plot 
`||query − mean(R)||` for **baseline** and **FM** features.
A clean detector should produce a step-up at `event_start` and stay high 
inside the alert window.

In [ ]:
# Switch the example by editing this single line.
EXAMPLE_SID = EXAMPLE_FIRE_SID   # try EXAMPLE_WIND_SID / EXAMPLE_THIN_SID / EXAMPLE_CLEAR_SID
ts_example = s2t.get_sample(ROOT, EXAMPLE_SID, metadata=meta)

emb_per_frame = emb_centre_for(EXAMPLE_SID)
pix_per_frame = pix_centre_for(EXAMPLE_SID)
print(f"Per-frame features: emb {emb_per_frame.shape}, pix {pix_per_frame.shape}")

queries = list(s2t.iter_detection_queries(ts_example))
dates_dt = [dt.date.fromisoformat(d) for d in ts_example.dates]
qd_arr, an_emb, an_pix, lbl_arr = [], [], [], []
for qi, lbl, ref in queries:
    qd_arr.append(dates_dt[qi])
    an_emb.append(np.linalg.norm(emb_per_frame[qi] - emb_per_frame[ref].mean(0)))
    an_pix.append(np.linalg.norm(pix_per_frame[qi] - pix_per_frame[ref].mean(0)))
    lbl_arr.append(lbl)
qd_arr = np.asarray(qd_arr); an_emb = np.asarray(an_emb)
an_pix = np.asarray(an_pix); lbl_arr = np.asarray(lbl_arr)
print(f"Eligible queries: {len(queries)} ({lbl_arr.sum()} positive)")

ev_start = ts_example.event_period()["start"]
fig = s2t.anomaly_score_timeseries(
    ts_example,
    {
        "Baseline (single centre pixel, 12 bands + 3 indices)": an_pix,
        "FM (TerraMind-small, single centre token)": an_emb,
    },
    query_dates=list(qd_arr),
    labels=lbl_arr,
    event_start=ev_start,
    alert_months=s2t.ALERT_WINDOW_MONTHS,   # 12 months by default
    figsize=(11, 5.0),                      # adjust to taste
    title=f"Anomaly score over time — sid={EXAMPLE_SID}",
)
plt.show()

### Build the training set across all 200 samples

Now let's build our training set for the NRT detection task.

> ⏱ The next cell loops over all 200 samples and assembles the recipe
> feature vector for every eligible query. **Expect ~30 s on CPU**.

In [ ]:
# One call to assemble (FM, baseline, label, sample_id) for every
# eligible (query, reference) tuple across all 200 samples. See
# `s2t.build_detection_dataset` for the underlying loop.
result = s2t.build_detection_dataset(
    meta,
    root=ROOT,
    emb_centre_for=emb_centre_for,
    pix_centre_for=pix_centre_for,
    recipe=RECIPE,
    n_ref_years=N_REF_YEARS,
    ref_window_days=REF_WINDOW_DAYS,
    alert_months=ALERT_MONTHS,
)
X_emb   = result["X_emb"]       # (N, recipe_dim_for_FM)
X_pix   = result["X_pix"]       # (N, recipe_dim_for_baseline)
y       = result["y"]           # (N,)  0/1 detection label
groups  = result["groups"]      # (N,)  sample_id — used for group-split CV

### Train a model on a single 80/20 group split

We can now train a simple classifier model on top of the dataset we prepared
and see how FM and baseline features perform.

In our below example, our **default classifier is a torch 
`LogisticRegression`-equivalent**: 1 linear layer supervised with 
class-weighted cross-entropy. However, it would be possible to define other 
models, of course. For instance, you can try using MLP in the 👩‍💻 Exercise below.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import f1_score

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=21)
tr, va = next(gss.split(X_emb, y, groups=groups))


def fit_eval(X_train, y_train, X_val, y_val, *, name,
             kind="lr", lr=5e-3, wd=1e-1, **model_kwargs):
    '''Fit a TorchClassifier on CPU and print train/val macro-F1 + gap.'''
    clf = TorchClassifier(kind, epochs=200, lr=lr, weight_decay=wd,
                          device=DEVICE, **model_kwargs)
    clf.fit(X_train, y_train)
    tr_f1 = f1_score(y_train, clf.predict(X_train), average="macro")
    va_f1 = f1_score(y_val,   clf.predict(X_val),   average="macro")
    print(f"  {name:32s}  train={tr_f1:.3f}  val={va_f1:.3f}  gap={tr_f1 - va_f1:+.3f}")
    return clf


print(f"single 80/20 group-split, seed=21 — train={len(tr)} val={len(va)}")
clf_fm_lr  = fit_eval(X_emb[tr], y[tr], X_emb[va], y[va], name="FM       LR",  kind="lr")
clf_bl_lr  = fit_eval(X_pix[tr], y[tr], X_pix[va], y[va], name="Baseline LR",  kind="lr")

### 👩‍💻 Exercise — non-blocking

Here you can use an MLP on top of the prepared features, rather than a simple 
linear layer. Feel free to play with it.

In [ ]:
# Exercise (optional): a non-linear MLP head on the same features.
# Audit shows MLP picks up ~0.03-0.05 val macro-F1 over LR on FM `full`,
# at the cost of a ~0.05-0.10 train-vs-val overfit gap. Try it:
clf_fm_mlp = fit_eval(
    X_emb[tr], y[tr], X_emb[va], y[va],
    name="FM       MLP (hidden=256)", kind="mlp",
    wd=1e-2, hidden=256, dropout=0.1,
)
clf_bl_mlp = fit_eval(
    X_pix[tr], y[tr], X_pix[va], y[va],
    name="Baseline MLP (hidden=256)", kind="mlp",
    wd=1e-2, hidden=256, dropout=0.1,
)

# Other knobs worth trying in additional cells below:
#  - swap RECIPE to "cd_pseudo_std", "diff", or "query_mean_cd"
#  - swap N_REF_YEARS in {1, 2, 3, 4}, REF_WINDOW_DAYS in {15, 30, 45, 90, 180}
#  - swap the distance metric inside compute_recipe (Euclidean / cosine /
#    per-element / Mahalanobis with reference covariance)
#  - bump to 5-fold CV (re-use sklearn.model_selection.GroupKFold)

### Dense temporal sweep on the example sample

The classifiers above were trained on **single-pixel / single-token** features
at the labelled centre. But the same trained head can be evaluated at *every*
spatial location — applying it across the whole patch gives us a per-pixel /
per-token *probability map*. Running this for a sequence of frames produces a
**temporal sweep**: you see the predicted disturbance spreading or fading
across space and time on a single sample.

> ⏱ The next cell evaluates both heads at 14 × 14 spatial locations for
> the 6 chosen frames. Expect ~20-30 s on CPU.

In [ ]:
# Switch the example by editing this single line.
EXAMPLE_SID = EXAMPLE_FIRE_SID   # try EXAMPLE_WIND_SID / EXAMPLE_THIN_SID / EXAMPLE_CLEAR_SID
ts_example = s2t.get_sample(ROOT, EXAMPLE_SID, metadata=meta)

# Apply the trained heads at every spatial location of `n_frames`
# evenly-spaced eligible frames (half before / half after the event).
# Internally this calls s2t.dense_token_features (FM) and
# s2t.dense_pixel_features (baseline) under the hood; see
# s2t.dense_detection_sweep for the orchestration.
fig, sweep_idx = s2t.dense_detection_sweep(
    ts_example,
    clf_fm=clf_fm_lr,
    clf_bl=clf_bl_lr,
    recipe=RECIPE,
    n_frames=6,           # how many frames to plot
    fig_width=14.0,       # adjust to taste
)
print("Sweep frames:", [ts_example.dates[i] for i in sweep_idx])
plt.show()

### 👩‍💻 Exercise - Try the reference-window knobs yourself

We invite you to play with other hyperparameters of the approach described
in this section. For instance, `N_REF_YEARS` and `REF_WINDOW_DAYS` are
important design choices that might affect model performance:

- `N_REF_YEARS` — how many past years contribute to the reference set $R$.
  Try `1`, `2`, `3`, `4`. Lower → more recent references, sharper
  contrast but fewer frames. Higher → more frames, but mixes in years
  with different growing conditions.
- `REF_WINDOW_DAYS` — calendar half-window around the query date.
  Try `15`, `30`, `45`, `60`, `90`, `180`. Tight windows give a cleaner
  same-season comparison; wide windows tolerate more revisit variance.

To see the impact, change one knob in the **Tunable knobs** cell, then
re-run the training-set build and the 80/20-split training cell. Watch
how `train_macro_F1`, `val_macro_F1`, and the gap move. The bigger the
gap, the harder the recipe is asking the head to memorise.

## 5. Task B — NRT Attribution (4-class agent assignment)

<p align="center">
  <img src="assets/nrt_attribution.png" alt="NRT attribution recipe" width="80%">
</p>

#### 🎯 Attribution task
Given that a _disturbance is known to have occurred_,
classify which agent caused it: **Clear-Cut (211)**, **Thinning (212)**,
**Wildfire (242)**, **Wind (243)**. 

**📏 Metric**: this is a multi-class classification task with class 
imbalance. We will measure performance by macro-averaged **F1 score**.

For this task, we propose another recipe for building temporal features
capturing the scene before and after the event (5 sub-vectors concatenated):

- `pre_mean`, `pre_std` — reference set ±`REF_WINDOW_DAYS` over past
  `N_REF_YEARS`, centred on `event_start` (the disturbance moment).
- `post_mean`, `post_std` — cloud-free frames in `[event_start, event_start + POST_EVENT_WINDOW_MONTHS]`.
- `post_mean − pre_mean` — the explicit "delta" channel.

Since our dataset has only 200 samples for this NRT attribution task, 
we do 5-fold stratified cross-validation to
capture more robust `mean ± std` of performance metrics.

### Build the attribution dataset

Let's build our training dataset. One vector per sample time series. 
We re-use the precomputed centre features from Section 3.

> ⏱ The cell below iterates over all 200 samples to assemble the
> attribution feature matrix. Expect ~10 s on CPU.

In [ ]:
# 4-class attribution palette — codes + human-readable names + plotting colors.
ATTR_CODES = np.array(s2t.DEFAULT_ATTR_CODES)
ATTR_NAMES = list(s2t.DEFAULT_ATTR_NAMES)
ATTR_TO_IDX = {int(c): i for i, c in enumerate(ATTR_CODES)}

# One call to assemble (FM, baseline, class label, sample_id) per sample.
# The 5-block per-sample recipe is
#     [pre_mean, pre_std, post_mean, post_std, post_mean - pre_mean]
# where the pre / post sets come from
#   pre  = past-year same-calendar references for event_start
#          (s2t.references_at_event)
#   post = cloud-free frames in
#          [event_start, event_start + POST_EVENT_WINDOW_MONTHS]
#          (s2t.post_event_indices).
result = s2t.build_attribution_dataset(
    meta,
    root=ROOT,
    emb_centre_for=emb_centre_for,
    pix_centre_for=pix_centre_for,
    attr_codes=ATTR_CODES,
    attr_names=ATTR_NAMES,
)
Xa_emb    = result["X_emb"]    # (N, 5 * 384)
Xa_pix    = result["X_pix"]    # (N, 5 * 15)
y_attr    = result["y"]        # (N,) int  index into ATTR_CODES
sids_attr = result["sids"]     # (N,) int

### Train a model with 5-fold stratified CV

Let's train on the cross-validation folds, stratified-by-class to keep 
the rare classes (Thinning 24, Wind 34) represented in every fold. 

We use the same `TorchClassifier` as before to get started, but you can 
also try a non-linear MLP later on.

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import confusion_matrix


def stratified_cv_attribution(X, y, *, n_splits=5, kind="lr",
                              lr=5e-3, wd=1e-1, **model_kwargs):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    tr_f1s, va_f1s, cm_sum = [], [], np.zeros((len(ATTR_CODES), len(ATTR_CODES)), dtype=int)
    last_clf = None
    for tr, va in skf.split(X, y):
        clf = TorchClassifier(kind, epochs=200, lr=lr, weight_decay=wd,
                              device=DEVICE, **model_kwargs)
        clf.fit(X[tr], y[tr])
        last_clf = clf
        yp_tr = clf.predict(X[tr]); yp_va = clf.predict(X[va])
        tr_f1s.append(f1_score(y[tr], yp_tr, average="macro"))
        va_f1s.append(f1_score(y[va], yp_va, average="macro"))
        cm_sum += confusion_matrix(y[va], yp_va, labels=np.arange(len(ATTR_CODES)))
    return np.asarray(tr_f1s), np.asarray(va_f1s), cm_sum, last_clf


print("5-fold stratified CV — FM:")
tr_fm, va_fm, cm_fm, clf_attr_fm = stratified_cv_attribution(Xa_emb, y_attr, kind="lr")
print(f"  FM       LR  train={tr_fm.mean():.3f}  val={va_fm.mean():.3f} ± {va_fm.std():.3f}")

print("5-fold stratified CV — Baseline:")
tr_bl, va_bl, cm_bl, clf_attr_bl = stratified_cv_attribution(Xa_pix, y_attr, kind="lr")
print(f"  Baseline LR  train={tr_bl.mean():.3f}  val={va_bl.mean():.3f} ± {va_bl.std():.3f}")
print(f"\nChance level (1/4):  {1/len(ATTR_CODES):.3f}")

### Confusion matrices

Let's look at the event attribution performances. 
How good are we for each class?
Where are our false positives and our false negatives?

In [ ]:
fig = s2t.confusion_matrices_figure(
    cms=[cm_bl, cm_fm],
    names=[
        "Baseline LR (centre pixel, 12 bands + 3 indices)",
        "FM LR (TerraMind-small centre token)",
    ],
    f1_arrays=[va_bl, va_fm],
    class_names=ATTR_NAMES,
    figsize=(10, 4),     # adjust to taste
)
plt.show()

### 👩‍💻 Exercise — optional non-linear head + recipe knobs

Here you can train an MLP for the NRT attribution task. See if it changes performance.

In [ ]:
# Exercise: torch MLP on the same dataset.
tr_fm_mlp, va_fm_mlp, _, _ = stratified_cv_attribution(
    Xa_emb, y_attr, kind="mlp", wd=1e-2, hidden=256, dropout=0.1,
)
print(f"  FM       MLP  train={tr_fm_mlp.mean():.3f}  "
      f"val={va_fm_mlp.mean():.3f} ± {va_fm_mlp.std():.3f}")

# Other knobs worth trying:
#  - shrink the post-event window (POST_EVENT_WINDOW_MONTHS = 1 vs 3 vs 6).
#  - replace mean+max by median + 90th-percentile + quartile-spread.
#  - reduce the recipe to {pre_mean, post_mean, delta} → 3 blocks, 1152-d.
#  - bump CV to 10 folds for tighter mean ± std estimates.

### Combining both models for dense attribution maps

We apply the trained 4-class attribution head at *every* spatial location of
the post-event frame. On its own, that produces a class assignment everywhere
— but the attribution head was trained to **assume** a disturbance, so it
will happily label any pixel as one of the four agents. To make the figure
honest we intersect attribution with the **detection** model at the same
location: only pixels whose detection probability clears
`DETECTION_THRESHOLD` are coloured with their attribution class; the rest
render black (no-alarm, no-attribution).

Each model gates with **its own** detector (FM × FM, baseline × baseline).
The post-event frame's references are the past-year same-calendar set
that the detection head already expects.

Switch the example by editing one line — the figure rebuilds for that
class. A coloured circle on the post-event RGB marks the annotated
pixel; its edge colour = the true class.

In [ ]:
# Switch the example by editing this single line.
EXAMPLE_SID = EXAMPLE_FIRE_SID   # try EXAMPLE_WIND_SID / EXAMPLE_THIN_SID / EXAMPLE_CLEAR_SID

# Single tunable: how confident detection must be to allow attribution.
# Lower -> more pixels coloured; higher -> only high-confidence alarms.
DETECTION_THRESHOLD = 0.85

# Run detection + attribution densely on the example sample and render
# the 4-panel figure. See s2t.dense_attribution_map for the gating
# logic (each model uses its own detector — FM x FM, baseline x baseline).
ts_example = s2t.get_sample(ROOT, EXAMPLE_SID, metadata=meta)
fig = s2t.dense_attribution_map(
    ts_example,
    clf_fm_det=clf_fm_lr,        # detection heads from Section 3
    clf_bl_det=clf_bl_lr,
    clf_fm_attr=clf_attr_fm,     # attribution heads from earlier in Section 4
    clf_bl_attr=clf_attr_bl,
    attr_codes=ATTR_CODES,
    attr_names=ATTR_NAMES,
    attr_palette=["#4c72b0", "#dd8452", "#dc2626", "#55a868"],
    attr_to_idx=ATTR_TO_IDX,
    threshold=DETECTION_THRESHOLD,
    det_recipe=RECIPE,
    fig_width=14.0,
    fig_height=3.6,
)
plt.show()

### 👩‍💻 Exercise - Try the attribution knobs yourself

The strongest levers on the attribution 5-fold scoreboard are:

- **`RECIPE`** (Section 3's tunable knob, re-used here for the per-frame
  features that feed the attribution head) — moving from `cd_scalar`
  (1 dim) to `full` ($4D$) is mostly a story of *capacity*: the FM head
  benefits from extra dims, the baseline head plateaus quickly.
- **`N_REF_YEARS`** — how far back to look. Going from `1` to `2` years
  buys you a more robust `pre_mean` / `pre_std`; beyond that the gain
  is small.
- **Post-event aggregation** — the recipe currently uses
  `[post_mean, post_std]`. Swap `post_std` for `post.max(0)` or
  `np.percentile(post, 90, axis=0)` inside
  `s2tutorial.nrt.attribution_feature_vector` to see how a different
  aggregator behaves on the same data.
- **Head choice** — replace the LR with an MLP (`kind="mlp"`,
  `hidden=256`, `dropout=0.1`) in `stratified_cv_attribution`. With
  only 200 samples the gain is modest; the gap usually grows.

Pick one knob, re-run **Build the attribution dataset** + the
**5-fold CV** cell, and compare `mean ± std` macro-F1.

## 6. Recap

### Headline scoreboard

Numbers below are **computed live in this notebook** — no reference to
pre-cached benchmark runs. Detection uses the single 80/20 group split
trained in Section 3; attribution uses the 5-fold stratified CV trained
in Section 4. Both heads are linear (`TorchClassifier("lr")`) on the
centre-feature recipe.

In [ ]:
print("=" * 56)
print(f"{'Task':<26s}{'mode':<10s}{'val macro-F1':>20s}")
print("-" * 56)
det_rows = [
    ("Detection (1×80/20)",  "FM LR",       f"{f1_score(y[va], clf_fm_lr.predict(X_emb[va]), average='macro'):.3f}"),
    ("",                      "Baseline LR", f"{f1_score(y[va], clf_bl_lr.predict(X_pix[va]), average='macro'):.3f}"),
]
attr_rows = [
    ("Attribution (5-fold)",  "FM LR",       f"{va_fm.mean():.3f} ± {va_fm.std():.3f}"),
    ("",                      "Baseline LR", f"{va_bl.mean():.3f} ± {va_bl.std():.3f}"),
]
for task, mode, score in det_rows + attr_rows:
    print(f"{task:<26s}{mode:<10s}{score:>20s}")
print("=" * 56)
print(f"\nChance (binary detection):    0.500")
print(f"Chance (4-class attribution): 0.250")

### 👩‍💻 Exercise - Things to try

Concrete experiments that fit naturally on top of what's above. Each
one is a 1-2 line edit, and re-execution from the relevant cell takes
under a minute on CPU.

**On the feature recipe** (`s2t.RECIPE_REGISTRY` exposes 6 named options)
- Swap `RECIPE` to `"diff"`, `"cd_pseudo_std"`, or `"query_mean_cd"`.
  How does the val macro-F1 move? Which recipe favours FM vs baseline?
- Replace the per-element delta inside `compute_recipe` with a
  cosine-distance or Mahalanobis-with-reference-covariance feature.

**On the reference set**
- Sweep `N_REF_YEARS` ∈ {1, 2, 3, 4} (re-run the dataset build cell).
- Sweep `REF_WINDOW_DAYS` ∈ {15, 30, 45, 90, 180}. Tight calendar
  windows give a cleaner same-season comparison; wider windows tolerate
  more revisit variance.
- Tighten / widen `POST_EVENT_WINDOW_MONTHS` for the attribution task —
  too short means missing the canopy-collapse signal; too long mixes in
  revegetation.

**On the classifier**
- Drop in a different `nn.Module` via `TorchClassifier` (any layer that
  accepts `(B, F)` input and produces `(B, K)` logits will work).
- Tune the MLP — `hidden`, `dropout`, `weight_decay`, `lr`.
- Run 5-fold CV on the detection task (set `N_SPLITS = 5` and use
  `GroupKFold` instead of `GroupShuffleSplit`) to get a `mean ± std`
  number for the headline scoreboard above.

**On evaluation**
- Move the `DETECTION_THRESHOLD` knob in the dense attribution section
  between `0.4` and `0.9` to trade precision for recall on the gated
  attribution map.
- Sweep a decision threshold on the FM detector's `P(event)` to control
  precision / recall trade-off — useful for an operational alert system
  where false alarms cost more than missed detections.
- Apply the trained 4-class attribution head to detection-positive
  samples only (instead of every sample), and report a *combined*
  detect-then-attribute score.

**On the data pipeline**
- Use `s2t.ForestDisturbanceDataModule` (raw S2) or
  `ForestDisturbanceEmbeddingDataModule` (embeddings) with a real
  Lightning trainer — the centre-feature aggregation above is what
  `sample_mode="frames"` + a head on top would produce.
- Re-generate the centre-pixel features at a non-centre location, or
  add a small spatial window (3×3 mean) — does it improve attribution?